# Pipeline NISAR → Douala (tâche 4 : notebook Colab)

**Projet** : NASA Space Apps 2026 — *Dancing with the SARs* (Douala / estuaire du Wouri)

Ce notebook : recherche les produits **NISAR GCOV** (bande L) sur Douala → les lit en streaming et les recadre sur la zone d'étude → calcule la rétrodiffusion en dB → produit le masque d'eau et les statistiques → exporte des **PNG + `manifest.json`** conformes à `docs/CONTRAT_DONNEES.md`.

**Avant de commencer**
1. Compte Earthdata + jeton : voir `docs/ACCES_EARTHDATA.md`.
2. Colab → icône 🔑 **Secrets** → créer `EARTHDATA_TOKEN` (valeur = votre jeton) et activer « Accès au notebook ».
3. Exécuter les cellules dans l'ordre. Si Colab demande de redémarrer après l'installation, faites-le puis reprenez à la cellule 2.

Résultat : un dossier `output/` (à copier dans `frontend/public/data/`).

In [ ]:
# 1. Installation (≈ 1-2 min)
!pip -q install earthaccess asf_search rasterio pyproj h5netcdf h5py xarray pillow matplotlib

In [ ]:
# 2. CONFIGURATION — constantes du CONTRAT DE DONNÉES (ne pas modifier sans accord de l'équipe)
import os, json, re, math, shutil, datetime as dt
from pathlib import Path
import numpy as np

SCHEMA_VERSION = "1.0"
PROJECT        = "dancing-with-the-sars-douala"
AOI_NAME       = "Douala – estuaire du Wouri"
BBOX           = (9.55, 3.90, 9.90, 4.20)   # (ouest, sud, est, nord) en degrés, EPSG:4326
IMAGE_SIZE     = (1400, 1200)               # (largeur, hauteur) en pixels, identique pour TOUS les PNG
DB_RANGE       = (-30.0, 0.0)               # échelle de gris des PNG de rétrodiffusion
THRESHOLD_DB   = -18.0                      # seuil d'eau par défaut (HH, dB) — hypothèse à valider (cellule 9)
MIN_VALID_FRACTION = 0.90                   # en dessous : acquisition marquée "partial"
WATER_RGBA     = (34, 211, 238, 170)        # cyan
NEW_WATER_RGBA = (251, 146, 60, 190)        # orange

# --- Paramètres propres à cette exécution (modifiables) ---
START_DATE = "2025-11-01"
END_DATE   = dt.date.today().isoformat()
MAX_ACQUISITIONS = 8                        # nombre max de dates retenues
SELECTED_DATES = None                       # ex. ["2026-01-15", "2026-07-26"] pour forcer le choix
OUT_DIR = Path("/content/output") if Path("/content").exists() else Path("output")

# Collections NISAR GCOV, de la plus mature à la moins mature
SHORT_NAMES = [
    ("NISAR_L2_GCOV_V1", "VALIDATED"),
    ("NISAR_L2_GCOV_PROVISIONAL_V1", "PROVISIONAL"),
    ("NISAR_L2_GCOV_BETA_V1", "BETA"),
]
MATURITY_RANK = {"VALIDATED": 0, "PROVISIONAL": 1, "BETA": 2}

WEST, SOUTH, EAST, NORTH = BBOX
WIDTH, HEIGHT = IMAGE_SIZE
print("Zone :", AOI_NAME, BBOX, "| image", IMAGE_SIZE, "| sortie :", OUT_DIR)

In [ ]:
# 3. AUTHENTIFICATION EARTHDATA (jeton stocké dans les Secrets Colab, jamais dans le notebook)
import earthaccess

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

token = get_secret("EARTHDATA_TOKEN")
if token:
    os.environ["EARTHDATA_TOKEN"] = token
else:
    user, pwd = get_secret("EARTHDATA_USERNAME"), get_secret("EARTHDATA_PASSWORD")
    if user and pwd:
        os.environ["EARTHDATA_USERNAME"], os.environ["EARTHDATA_PASSWORD"] = user, pwd

try:
    auth = earthaccess.login(strategy="environment")
except Exception as exc:
    print("Connexion par variables d'environnement impossible :", exc)
    auth = earthaccess.login(strategy="interactive")   # saisie manuelle

print("Authentifié :", bool(getattr(auth, "authenticated", False)))

# Test d'accès : une recherche + lecture de la signature du fichier HDF5
test = earthaccess.search_data(short_name="NISAR_L2_GCOV_PROVISIONAL_V1", bounding_box=BBOX, count=1)
print("Granules trouvés pour le test :", len(test))
if test:
    f = earthaccess.open(test[:1])[0]
    print("Signature du fichier :", f.read(8))   # doit contenir b'HDF'

In [ ]:
# 4. RECHERCHE des acquisitions NISAR GCOV (bande L) sur la zone
def parse_name(ur):
    # NISAR_L2_PR_GCOV_CYL_REL_P_FRM_MODE_POLE_S_START_END_...
    t = ur.replace(".h5", "").split("_")
    try:
        return {"band": t[1], "track": t[5], "orbit": t[6], "pol_code": t[9]}
    except IndexError:
        return {"band": None, "track": None, "orbit": None, "pol_code": None}

def parse_start(g):
    s = g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    return dt.datetime.fromisoformat(s.replace("Z", "+00:00")).astimezone(dt.timezone.utc)

found = []
for short_name, maturity in SHORT_NAMES:
    try:
        res = earthaccess.search_data(short_name=short_name, bounding_box=BBOX,
                                      temporal=(START_DATE, END_DATE), count=300)
    except Exception as exc:
        print(f"[{maturity}] recherche impossible :", exc); continue
    print(f"[{maturity}] {len(res)} granule(s)")
    for g in res:
        ur = g["umm"]["GranuleUR"]
        info = parse_name(ur)
        if info["band"] not in (None, "L2"):     # on garde la bande L (instrument "L", niveau 2)
            continue
        start = parse_start(g)
        found.append({"granule": g, "ur": ur, "maturity": maturity, "start": start,
                      "date": start.date().isoformat(), **info})

print("Total :", len(found), "| dates distinctes :", len({f["date"] for f in found}))
for f in sorted(found, key=lambda r: r["start"])[:40]:
    print(f["date"], f["maturity"], f["orbit"], f["track"], f["pol_code"], f["ur"][:60])

In [ ]:
# 5. SÉLECTION : une acquisition par date, même orbite/trace (comparabilité), dates espacées
def season_of(date_str):
    return "dry" if int(date_str[5:7]) in (12, 1, 2) else "rain"

# Une seule entrée par date (la plus mature)
by_date = {}
for f in sorted(found, key=lambda r: (MATURITY_RANK[r["maturity"]], r["start"])):
    by_date.setdefault(f["date"], f)
cands = list(by_date.values())

# Garder le groupe (sens d'orbite, trace) le plus fourni : séries temporelles cohérentes
groups = {}
for f in cands:
    groups.setdefault((f["orbit"], f["track"]), []).append(f)
if groups:
    best = max(groups, key=lambda k: len(groups[k]))
    cands = sorted(groups[best], key=lambda r: r["date"])
    print("Groupe retenu (orbite, trace) :", best, "->", len(cands), "date(s)")

if SELECTED_DATES:
    selected = [f for f in cands if f["date"] in SELECTED_DATES]
elif len(cands) > MAX_ACQUISITIONS:
    idx = np.unique(np.linspace(0, len(cands) - 1, MAX_ACQUISITIONS).round().astype(int))
    selected = [cands[i] for i in idx]
    # s'assurer d'avoir une référence saison sèche si elle existe
    dry = [f for f in cands if season_of(f["date"]) == "dry"]
    if dry and not any(season_of(f["date"]) == "dry" for f in selected):
        selected[0] = dry[0]
else:
    selected = cands
selected = sorted({f["date"]: f for f in selected}.values(), key=lambda r: r["date"])

print("\nAcquisitions retenues :")
for f in selected:
    print(" ", f["date"], season_of(f["date"]), f["maturity"], f["orbit"], f["track"])
if not any(season_of(f["date"]) == "dry" for f in selected):
    print("\n⚠ Aucune acquisition en saison sèche (déc.-fév.) : pas de référence « eau permanente » fiable.")

In [ ]:
# 6. LECTURE + RECADRAGE + REPROJECTION (streaming : seule la zone utile est lue)
import xarray as xr
from pyproj import Transformer
from rasterio.crs import CRS
from rasterio.transform import from_origin, from_bounds
from rasterio.warp import reproject, Resampling

GRID_GROUP = "/science/LSAR/GCOV/grids/frequencyA"
DST_TRANSFORM = from_bounds(WEST, SOUTH, EAST, NORTH, WIDTH, HEIGHT)

def _span(coord, lo, hi):
    inside = np.where((coord >= lo) & (coord <= hi))[0]
    if inside.size == 0:
        return None
    return max(int(inside[0]) - 1, 0), min(int(inside[-1]) + 2, coord.size)

def _find(ds, names):
    for n in names:
        if n in ds.variables:
            return ds[n]
    return None

def _epsg(ds, lon):
    try:
        return int(np.asarray(ds["projection"].values).item())
    except Exception:
        pass
    try:
        return int(ds["projection"].attrs["epsg_code"])
    except Exception:
        zone = int((lon + 180) // 6) + 1
        print("EPSG non trouvé dans le fichier, UTM nord supposé : zone", zone)
        return 32600 + zone

def read_aoi(ds, var):
    # Retourne la puissance (linéaire) reprojetée en EPSG:4326 sur la grille du contrat, ou None.
    if var not in ds.variables:
        return None
    da = ds[var]
    ydim, xdim = da.dims[-2:]
    xc = _find(ds, ["xCoordinates", xdim]); yc = _find(ds, ["yCoordinates", ydim])
    if xc is None or yc is None:
        raise ValueError("Coordonnées x/y introuvables : inspecter ds.variables")
    x = np.asarray(xc.values, dtype="float64"); y = np.asarray(yc.values, dtype="float64")
    epsg = _epsg(ds, 0.5 * (WEST + EAST))
    tr = Transformer.from_crs(4326, epsg, always_xy=True)
    xs, ys = tr.transform([WEST, EAST, WEST, EAST], [SOUTH, SOUTH, NORTH, NORTH])
    sx, sy = _span(x, min(xs), max(xs)), _span(y, min(ys), max(ys))
    if sx is None or sy is None:
        return None
    (j0, j1), (i0, i1) = sx, sy
    arr = np.asarray(da.isel({ydim: slice(i0, i1), xdim: slice(j0, j1)}).values, dtype="float32")
    dx, dy = abs(float(np.median(np.diff(x)))), abs(float(np.median(np.diff(y))))
    if y[0] < y[-1]:                      # axe y croissant : on remet le nord en haut
        arr = arr[::-1]; ytop = y[i1 - 1]
    else:
        ytop = y[i0]
    arr[~np.isfinite(arr) | (arr <= 0)] = np.nan
    src_transform = from_origin(x[j0] - dx / 2, ytop + dy / 2, dx, dy)
    dst = np.full((HEIGHT, WIDTH), np.nan, dtype="float32")
    reproject(source=arr, destination=dst, src_transform=src_transform, src_crs=CRS.from_epsg(epsg),
              dst_transform=DST_TRANSFORM, dst_crs=CRS.from_epsg(4326),
              src_nodata=np.nan, dst_nodata=np.nan, resampling=Resampling.bilinear)
    return dst

def open_gcov(granule):
    fobj = earthaccess.open([granule])[0]
    tree = xr.open_datatree(fobj, engine="h5netcdf", decode_timedelta=False, phony_dims="access")
    return tree, fobj

def to_db(lin):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10.0 * np.log10(lin)

In [ ]:
# 7. TRAITEMENT : lecture de chaque acquisition retenue (peut prendre quelques minutes)
grids = {}   # date -> {"hh": dB, "hv": dB|None, "meta": {...}}
for f in selected:
    print("→", f["date"], f["ur"][:70])
    try:
        tree, fobj = open_gcov(f["granule"])
        ds = tree[GRID_GROUP].to_dataset()
        hh = read_aoi(ds, "HHHH")
        if hh is None:
            print("   HH absent ou hors zone : ignoré"); continue
        hv = read_aoi(ds, "HVHV")
        grids[f["date"]] = {"hh": to_db(hh), "hv": None if hv is None else to_db(hv), "meta": f}
        valid = np.isfinite(grids[f["date"]]["hh"]).mean()
        print(f"   OK — couverture valide : {valid:.0%}")
        fobj.close()
    except Exception as exc:
        print("   ÉCHEC :", type(exc).__name__, exc)
print("\nDates lues :", sorted(grids))

In [ ]:
# 8. CONTRÔLE VISUEL d'une date
import matplotlib.pyplot as plt
d0 = sorted(grids)[0]
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(grids[d0]["hh"], cmap="gray", vmin=DB_RANGE[0], vmax=DB_RANGE[1]); ax[0].set_title(f"HH (dB) – {d0}")
ax[1].imshow(grids[d0]["hh"] < THRESHOLD_DB, cmap="Blues"); ax[1].set_title(f"HH < {THRESHOLD_DB} dB")
for a in ax: a.axis("off")
plt.show()

In [ ]:
# 9. SEUIL D'EAU : histogramme + suggestion d'Otsu (à discuter en équipe, puis figer THRESHOLD_DB en cellule 2)
def otsu_threshold(db, lo=-35.0, hi=5.0, bins=256):
    v = db[np.isfinite(db)]; v = v[(v >= lo) & (v <= hi)]
    hist, edges = np.histogram(v, bins=bins, range=(lo, hi))
    c = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist); w1 = w0[-1] - w0
    m0 = np.cumsum(hist * c); mt = m0[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        between = w0 * w1 * ((m0 / w0) - ((mt - m0) / w1)) ** 2
    return float(c[np.nanargmax(between)])

plt.figure(figsize=(8, 3.5))
for d, g in grids.items():
    v = g["hh"][np.isfinite(g["hh"])]
    plt.hist(v, bins=120, range=(-35, 5), histtype="step", label=d)
plt.axvline(THRESHOLD_DB, color="red", ls="--", label=f"seuil = {THRESHOLD_DB} dB")
plt.xlabel("HH (dB)"); plt.legend(fontsize=7); plt.show()
print("Seuil d'Otsu suggéré par date :", {d: round(otsu_threshold(g["hh"]), 1) for d, g in grids.items()})

In [ ]:
# 10. MASQUES, STATISTIQUES ET EXPORT (PNG + manifest.json conformes au contrat)
from PIL import Image

LAYERS_DIR = OUT_DIR / "layers"
LAYERS_DIR.mkdir(parents=True, exist_ok=True)

RES_X, RES_Y = (EAST - WEST) / WIDTH, (NORTH - SOUTH) / HEIGHT
lat_rows = NORTH - (np.arange(HEIGHT) + 0.5) * RES_Y
AREA_MAP = ((111.32 * np.cos(np.radians(lat_rows)) * RES_X) * (110.57 * RES_Y))[:, None] * np.ones((1, WIDTH))  # km² / pixel

def write_backscatter_png(db, path):
    lo, hi = DB_RANGE
    g = (np.clip((db - lo) / (hi - lo), 0, 1) * 255)
    ok = np.isfinite(db)
    g = np.where(ok, g, 0).astype(np.uint8)
    a = np.where(ok, 255, 0).astype(np.uint8)
    Image.fromarray(np.dstack([g, g, g, a]), "RGBA").save(path, optimize=True)

def write_mask_png(mask, rgba, path):
    out = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)
    out[mask] = rgba
    Image.fromarray(out, "RGBA").save(path, optimize=True)

def km2(mask):
    return float((AREA_MAP * mask).sum())

masks, stats = {}, {}
for d, g in grids.items():
    valid = np.isfinite(g["hh"])
    masks[d] = valid & (g["hh"] < THRESHOLD_DB)
    stats[d] = {"valid": valid, "valid_fraction": float(valid.mean())}

# Référence "eau permanente" : scène saison sèche la plus complète (sinon la scène la moins humide)
dry_dates = [d for d in grids if season_of(d) == "dry"]
if dry_dates:
    reference_id = max(dry_dates, key=lambda d: stats[d]["valid_fraction"])
else:
    reference_id = min(grids, key=lambda d: masks[d].sum())
    print("⚠ Pas de saison sèche : référence =", reference_id, "(à signaler dans le README)")
print("Référence eau permanente :", reference_id)

acquisitions = []
for d in sorted(grids):
    g, f = grids[d], grids[d]["meta"]
    valid, water = stats[d]["valid"], masks[d]
    layers = {"backscatter_hh": f"layers/{d}_backscatter_hh.png", "water_mask": f"layers/{d}_water_mask.png"}
    write_backscatter_png(g["hh"], OUT_DIR / layers["backscatter_hh"])
    write_mask_png(water, WATER_RGBA, OUT_DIR / layers["water_mask"])
    if g["hv"] is not None:
        layers["backscatter_hv"] = f"layers/{d}_backscatter_hv.png"
        write_backscatter_png(g["hv"], OUT_DIR / layers["backscatter_hv"])
    new_km2 = None
    if d != reference_id:
        new_mask = water & ~masks[reference_id] & stats[reference_id]["valid"]
        new_km2 = round(km2(new_mask), 3)
        layers["new_water_mask"] = f"layers/{d}_new_water_mask.png"
        write_mask_png(new_mask, NEW_WATER_RGBA, OUT_DIR / layers["new_water_mask"])
    vf = stats[d]["valid_fraction"]
    acquisitions.append({
        "id": d, "date": d,
        "datetime_utc": f["start"].strftime("%Y-%m-%dT%H:%M:%SZ"),
        "season": season_of(d),
        "orbit": f["orbit"],
        "polarizations": ["HH"] + (["HV"] if g["hv"] is not None else []),
        "product": f["ur"],
        "maturity": f["maturity"],
        "mock": False,
        "partial": bool(vf < MIN_VALID_FRACTION),
        "layers": layers,
        "stats": {
            "water_area_km2": round(km2(water), 3),
            "new_water_km2": new_km2,
            "water_fraction": round(float(water.sum() / max(valid.sum(), 1)), 4),
            "valid_fraction": round(vf, 4),
            "threshold_db": THRESHOLD_DB,
            "mean_hh_db": round(float(np.nanmean(g["hh"])), 2),
        },
    })

manifest = {
    "schema_version": SCHEMA_VERSION,
    "generated_at": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "project": PROJECT,
    "aoi": {"name": AOI_NAME, "bbox": list(BBOX), "crs": "EPSG:4326", "image_size": list(IMAGE_SIZE)},
    "db_range": list(DB_RANGE),
    "water_style": {"color": "#22d3ee", "opacity": 0.67},
    "threshold_default_db": THRESHOLD_DB,
    "min_valid_fraction": MIN_VALID_FRACTION,
    "reference_id": reference_id,
    "acquisitions": acquisitions,
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✔ {len(acquisitions)} acquisition(s) exportée(s) dans {OUT_DIR}")
for a in acquisitions:
    s = a["stats"]; print(" ", a["date"], a["season"], "eau =", s["water_area_km2"], "km²", "| partiel" if a["partial"] else "")

In [ ]:
# 11. LIVRAISON : archive à télécharger (puis dézipper dans frontend/public/data/) ou copie vers Google Drive
archive = shutil.make_archive("/tmp/douala_sar_data", "zip", OUT_DIR)
print("Archive :", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Hors Colab : récupérez l'archive manuellement.")

# Option Drive :
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copytree(OUT_DIR, "/content/drive/MyDrive/douala-sar-output", dirs_exist_ok=True)